# AdPulse Analytics Pipeline
## Marketing Campaign Data Analysis Using PySpark

**Datasets:**
- ad_campaigns_data.json
- user_profile_data.json  
- store_data.json

**Analysis:**
- Q1: Events by OS Type
- Q2: Events by Store Name
- Q3: Events by Gender

## Step 1: Setup & Imports

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AdPulse Analytics Pipeline") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Version:", spark.version)
print("✅ SparkSession created successfully!")

Spark Version: 3.5.5
✅ SparkSession created successfully!


## Step 2: Load Raw Data

In [2]:
campaigns_df = spark.read.json("/data/datasets/ad_campaigns_data.jsonl")
users_df     = spark.read.json("/data/datasets/user_profile_data.jsonl")
stores_df    = spark.read.json("/data/datasets/store_data.jsonl")



print("✅ Data loaded!")
print(f"Campaigns : {campaigns_df.count()} rows")
print(f"Users     : {users_df.count()} rows")
print(f"Stores    : {stores_df.count()} rows")

✅ Data loaded!
Campaigns : 4 rows
Users     : 6 rows
Stores    : 4 rows


In [3]:
print("=== CAMPAIGNS SCHEMA ===")
campaigns_df.printSchema()

=== CAMPAIGNS SCHEMA ===
root
 |-- campaign_country: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- os_type: string (nullable = true)
 |-- place_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
print("=== UERS SCHEMA ===")
users_df.printSchema()

=== UERS SCHEMA ===
root
 |-- age_group: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- country: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
print("=== Stores Schema ===")
stores_df.printSchema()

=== Stores Schema ===
root
 |-- place_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- store_name: string (nullable = true)



In [6]:
print("=== CAMPAIGNS DATA ===")
campaigns_df.show(truncate=False)

=== CAMPAIGNS DATA ===
+----------------+-----------+-----------------------------+-----------+------------------------+----------+-------+---------+-------------------+
|campaign_country|campaign_id|campaign_name                |device_type|event_time              |event_type|os_type|place_id |user_id            |
+----------------+-----------+-----------------------------+-----------+------------------------+----------+-------+---------+-------------------+
|USA             |ABCDFAE    |Food category target campaign|apple      |2018-10-12T13:10:05.000Z|impression|ios    |CASSBB-11|1264374214654454321|
|USA             |ABCDFAE    |Food category target campaign|MOTOROLA   |2018-10-12T13:09:04.000Z|impression|android|CADGBD-13|1674374214654454321|
|USA             |ABCDFAE    |Food category target campaign|SAMSUNG    |2018-10-12T13:10:10.000Z|video ad  |android|BADGBA-12|5747421465445443   |
|USA             |ABCDFAE    |Food category target campaign|SAMSUNG    |2018-10-12T13:10:12.000

In [7]:
print("=== USERS DATA ===")
users_df.show(truncate=False)

=== USERS DATA ===
+---------+-------------------------------+-------+------+-------------------+
|age_group|category                       |country|gender|user_id            |
+---------+-------------------------------+-------+------+-------------------+
|18-25    |[shopper, student]             |USA    |male  |1264374214654454321|
|25-50    |[parent]                       |USA    |female|1674374214654454321|
|25-50    |[shopper, parent, professional]|USA    |male  |5747421465445443   |
|50+      |[professional]                 |USA    |male  |1864374214654454132|
|18-25    |[shopper, student]             |USA    |female|14537421465445443  |
|50+      |[shopper, professional]        |USA    |female|25547421465445443  |
+---------+-------------------------------+-------+------+-------------------+



In [9]:
print("=== STORES DATA ===")
stores_df.show(truncate=False)


=== STORES DATA ===
+---------------------------------+-------------+
|place_ids                        |store_name   |
+---------------------------------+-------------+
|[CASSBB-11, CADGBD-13, FDBEGD-14]|McDonald     |
|[CASSBB-11]                      |BurgerKing   |
|[BADGBA-13, CASSBB-15, FDBEGD-15]|Macys        |
|[BADGBA-12]                      |shoppers stop|
+---------------------------------+-------------+



In [11]:
from pyspark.sql import functions as F

campaigns_with_time = campaigns_df\
                      .withColumn("date",F.date_format(F.col("event_time"),"yyyy-MM-dd"))\
                      .withColumn("hour",F.date_format(F.col("event_time"),"HH"))

campaigns_with_time.select("event_time","date","hour").show(truncate=False);

+------------------------+----------+----+
|event_time              |date      |hour|
+------------------------+----------+----+
|2018-10-12T13:10:05.000Z|2018-10-12|13  |
|2018-10-12T13:09:04.000Z|2018-10-12|13  |
|2018-10-12T13:10:10.000Z|2018-10-12|13  |
|2018-10-12T13:10:12.000Z|2018-10-12|13  |
+------------------------+----------+----+



In [15]:
q1_grouped = campaigns_with_time.groupBy("campaign_id","date","hour","os_type","event_type").agg(F.count("*").alias("event_count"))

q1_grouped.show(truncate=False)

+-----------+----------+----+-------+----------+-----------+
|campaign_id|date      |hour|os_type|event_type|event_count|
+-----------+----------+----+-------+----------+-----------+
|ABCDFAE    |2018-10-12|13  |ios    |impression|1          |
|ABCDFAE    |2018-10-12|13  |android|click     |1          |
|ABCDFAE    |2018-10-12|13  |android|impression|1          |
|ABCDFAE    |2018-10-12|13  |android|video ad  |1          |
+-----------+----------+----+-------+----------+-----------+



In [19]:
q1_event_map = q1_grouped.groupBy("campaign_id","date","hour","os_type").agg(F.map_from_entries(F.collect_list(F.struct(F.col("event_type"),F.col("event_count")))).alias("event"))

q1_event_map.show(truncate=False)

+-----------+----------+----+-------+--------------------------------------------+
|campaign_id|date      |hour|os_type|event                                       |
+-----------+----------+----+-------+--------------------------------------------+
|ABCDFAE    |2018-10-12|13  |ios    |{impression -> 1}                           |
|ABCDFAE    |2018-10-12|13  |android|{click -> 1, impression -> 1, video ad -> 1}|
+-----------+----------+----+-------+--------------------------------------------+



In [21]:
q1_final = q1_event_map.select(
           F.col("campaign_id"),
           F.col("date"),
           F.col("hour"),
           F.lit("os_type").alias("type"),
           F.col("os_type").alias("value"),
           F.col("event"))


q1_final.show(truncate=False)

+-----------+----------+----+-------+-------+--------------------------------------------+
|campaign_id|date      |hour|type   |value  |event                                       |
+-----------+----------+----+-------+-------+--------------------------------------------+
|ABCDFAE    |2018-10-12|13  |os_type|ios    |{impression -> 1}                           |
|ABCDFAE    |2018-10-12|13  |os_type|android|{click -> 1, impression -> 1, video ad -> 1}|
+-----------+----------+----+-------+-------+--------------------------------------------+



In [22]:
OUTPUT_Q1 = "/data/outputs/q1_os_type"

q1_final.coalesce(1).write.mode("overwrite").json(OUTPUT_Q1)

print(f" File Successfully saved to : {OUTPUT_Q1}")


 File Successfully saved to : /data/outputs/q1_os_type
